# CSV: matched-cost wedges and heterogeneity

Experimental notebook interface to `extension.clustering.mispricing.pipeline`. The existing implementation remains authoritative. Run cells from top to bottom in a fresh kernel. Outputs are isolated under `output/notebook_extension/`; rerunning replaces that notebook’s outputs.

This is a staged orchestration of existing functions, not an independent reimplementation. Open a function’s source to change its algorithm; substitute a compatible function call below to test a variation. Preserve train-only fitting, splits, matched costs, support thresholds and dependence-aware uncertainty. See the repository methodology before interpreting results.

## 1. Locate the project and import reusable functions
The Python environment must contain the project dependencies plus a Jupyter kernel.

In [ ]:
import os
import sys
from pathlib import Path

# Works when Jupyter starts in this folder or anywhere inside the repository.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "extension/pipeline.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the project repository.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/ds340-notebook-mpl")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/ds340-notebook-cache")
from extension.clustering.mispricing.pipeline import (  # noqa: E402 - repository path must be set before project imports
    Path,
    argparse,
    curve,
    estimate,
    extract,
    fingerprint,
    hashlib,
    importlib,
    json,
    pd,
    positions,
    subprocess,
    threadpool_limits,
)

SOURCE_FILE = ROOT / "extension/clustering/mispricing/pipeline.py"

## 2. Configuration
Defaults match the existing CLI, except for isolated paths. Edit this argument list for a separate sensitivity run, using a new output directory. Do not select a model using the held-out results.

In [ ]:
NOTEBOOK_ARGUMENTS = [
    "--data",
    "data/kalshi_recent_year_csv",
    "--assignments",
    "output/notebook_extension/clustering/assignments.csv",
    "--output",
    "output/notebook_extension/mispricing",
]

## Parse and validate settings

In [ ]:
parser = argparse.ArgumentParser(description=__doc__)
parser.add_argument("--data", type=Path, default=Path("data/kalshi_recent_year_csv"))
parser.add_argument("--assignments", type=Path, default=Path("output/clustering/assignments.csv"))
parser.add_argument("--output", type=Path, default=Path("extension/clustering/mispricing/results"))
parser.add_argument(
    "--reuse-stats", action="store_true", help="Reuse audited sufficient statistics only if input metadata match"
)
args = parser.parse_args(NOTEBOOK_ARGUMENTS)

# Keep experimental writes away from the established implementation and results.
prototype_root = (ROOT / "output/notebook_extension").resolve()
if prototype_root not in args.output.resolve().parents:
    raise ValueError("Choose an output subdirectory under output/notebook_extension/.")
args.output.mkdir(parents=True, exist_ok=True)
(args.output / "run_manifest.json").write_text(
    json.dumps({"completed": False, "interface": "experimental_notebook"}, indent=2)
)

## Create the isolated run directory

In [ ]:
args.output.mkdir(parents=True, exist_ok=True)

## Audit inputs and extract sufficient statistics

In [ ]:
inputs = fingerprint([args.data / "master/markets.csv", args.data / "master/trades.csv", args.assignments])
audit_path = args.output / "data_audit.json"
if args.reuse_stats and audit_path.exists():
    audit = json.loads(audit_path.read_text())
    if audit["inputs"] != inputs:
        raise ValueError("Cached statistics input metadata do not match")
    cells = pd.read_csv(args.output / "sufficient_statistics.csv.gz", keep_default_na=False)
else:
    print("Scanning CSVs and auditing trade IDs...", flush=True)
    (cells, audit) = extract(args.data, args.assignments, args.output)
    audit["inputs"] = inputs
    audit_path.write_text(json.dumps(audit, indent=2) + "\n")
print(f"{len(cells):,} contract-price sufficient-statistic rows", flush=True)

## Record analysis provenance

In [ ]:
manifest = {
    "completed": False,
    "inputs": inputs,
    "primary": "execution-weighted, pooled splits, costs 1-49",
    "config": vars(args).copy(),
    "source_hashes": {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(SOURCE_FILE).parent.glob("*.py")
    },
}
manifest["completed"] = False
manifest["notebook_source_sha256"] = hashlib.sha256(
    (ROOT / "notebooks/extension_prototype/03_csv_mispricing.ipynb").read_bytes()
).hexdigest()
manifest["interface"] = "experimental_notebook"
manifest["config"] = {k: str(v) if isinstance(v, Path) else v for (k, v) in manifest["config"].items()}
manifest["git_revision"] = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
manifest_path = args.output / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

## Define matched populations and compute pooled curves

In [ ]:
assigned = cells[cells.cluster.ne("")].copy()
pops = {"all_resolved": cells, "non_combination": cells[cells.non_combination.eq(True)], "assigned_matched": assigned}
coverage = []
curves = []
summaries = []
tests = []
for name, subset in pops.items():
    coverage.append(
        {
            "population": name,
            "contracts": subset.ticker.nunique(),
            "events": subset.event_id.nunique(),
            "executions": int(subset.n.sum()),
            "units": float(subset.units.sum()),
        }
    )
    for weighting, column in [("execution", "n"), ("unit", "units")]:
        c = curve(positions(subset, column))
        c["population"] = name
        c["weighting"] = weighting
        curves.append(c)
pd.DataFrame(coverage).to_csv(args.output / "population_coverage.csv", index=False)
pd.concat(curves, ignore_index=True).to_csv(args.output / "pooled_curves.csv", index=False)

## Estimate cluster wedges, uncertainty and structural-family contrasts

In [ ]:
with threadpool_limits(limits=2):
    for weighting, column in [("execution", "n"), ("unit", "units")]:
        print(f"Constructing {weighting}-weighted positions...", flush=True)
        pos = positions(assigned, column)
        c = curve(pos, True)
        c["weighting"] = weighting
        c.to_csv(args.output / f"cluster_curves_{weighting}.csv", index=False)
        for band, mincost in [((1, 49), 10), ((1, 10), 6), ((11, 20), 6), ((21, 30), 6), ((31, 40), 6), ((41, 49), 6)]:
            print(f"Inference: {weighting} costs {band}", flush=True)
            (rows, test) = estimate(pos, band, mincost, 20, 10, weighting)
            summaries.append(rows)
            tests.append(dict(weighting=weighting, reference="pooled", band=f"{band[0]}-{band[1]}", **test))
        print(f"Inference: {weighting} within structural families", flush=True)
        (rows, test) = estimate(pos, (1, 49), 10, 20, 10, weighting, reference="family")
        summaries.append(rows)
        tests.append(dict(weighting=weighting, reference="family", band="1-49", **test))
        split_curves = []
        for split, g in pos.groupby("split"):
            c = curve(g, True)
            c["split"] = split
            split_curves.append(c)
        pd.concat(split_curves).to_csv(args.output / f"split_curves_{weighting}.csv", index=False)

## Export inference tables

In [ ]:
pd.concat(summaries, ignore_index=True).to_csv(args.output / "cluster_summary.csv", index=False)
pd.DataFrame(tests).to_csv(args.output / "heterogeneity_tests.csv", index=False)

## Generate the isolated report and figures

In [ ]:
from extension.clustering.mispricing.report import write_report

write_report(args.output, args.output / "FINDINGS.md")

## Finalize the run

In [ ]:
manifest["versions"] = {
    name: importlib.metadata.version(name) for name in ["numpy", "pandas", "scipy", "duckdb", "matplotlib"]
}
manifest["fixed_support_equality_completed"] = True
manifest["report_sha256"] = hashlib.sha256((args.output / "FINDINGS.md").read_bytes()).hexdigest()
manifest["completed"] = True
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("Completed. Report: " + str(args.output / "FINDINGS.md"), flush=True)

## Inspect results and limitations
A completed manifest means execution finished; it does not validate the research assumptions. This is not a chronological backtest, a causal wording experiment, or an execution-aware trading simulation. Sparse/rejected groups must remain visible. Compare this run with the CLI using identical inputs, settings and software before adopting the notebook route.

In [ ]:
from IPython.display import Image, display

result_dir = args.output.resolve()
print("Results:", result_dir)
print("Completed:", json.loads((result_dir / "run_manifest.json").read_text()).get("completed", False))
for csv_name in ["model_comparison.csv", "coverage.csv", "population_coverage.csv", "heterogeneity_tests.csv"]:
    csv_path = result_dir / csv_name
    if csv_path.exists():
        print(csv_name)
        display(pd.read_csv(csv_path).head(20))
for image_path in sorted(result_dir.glob("*.png")) + sorted((result_dir / "figures").glob("*.png")):
    display(Image(filename=str(image_path)))
report_path = result_dir / "FINDINGS.md"
if not report_path.exists():
    report_path = result_dir / "STRUCTURAL_FINDINGS.md"
if report_path.exists():
    print("Report:", report_path)